In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image

PATCH = 350


def find_repo_with_ews(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "EWS" / "data").is_dir():
            return p
    raise FileNotFoundError(" EWS/data，。")


REPO = find_repo_with_ews(Path.cwd())
OUT_DIR = REPO / "EWS" / "data" / "npy"

IMG_DIR = (
    REPO
    / "EWS"
    / "data"
    / "sugar-beets-2016-DatasetNinja"
    / "ds"
    / "img_350_crop_ews_y"
)


OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO:", REPO)
print("IMG_DIR:", IMG_DIR)
print("OUT_DIR:", OUT_DIR)

In [ ]:
def png_to_patch_npy(src: Path, dst: Path) -> None:
    im = Image.open(src).convert("RGB")
    if im.size != (PATCH, PATCH):
        raise ValueError(f" {PATCH}×{PATCH}， {im.size[0]}×{im.size[1]}: {src.name}")
    arr = np.asarray(im, dtype=np.uint8)
    x = arr.astype(np.float32) / 255.0
    np.save(dst, x, allow_pickle=False)


if not IMG_DIR.is_dir():
    raise FileNotFoundError(f": {IMG_DIR}")

paths = sorted(IMG_DIR.glob("rgb_*.png"))
if not paths:
    raise FileNotFoundError(f" rgb_*.png: {IMG_DIR}")

print(f": {len(paths)} （1  .npy）")

In [ ]:
from tqdm import tqdm
it = tqdm(paths, desc="→ npy") if tqdm else paths
for p in it:
    out_path = OUT_DIR / f"{p.stem}.npy"
    png_to_patch_npy(p, out_path)

print(f": {len(paths)}  .npy → {OUT_DIR}")

In [ ]:
chk = OUT_DIR / f"{paths[0].stem}.npy"
a = np.load(chk, mmap_mode=None)
print(chk.name, a.shape, a.dtype, float(a.min()), float(a.max()))